In [0]:
#%skip
%pip install Faker

In [0]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import random
from faker import Faker

try:
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/churn_labels.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/connection_quality_logs.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/customer_profile.csv')
    os.remove ('/Workspace/Users/nina.merkt@abat.de/test/data/agent_profile.csv')
except FileNotFoundError:
    pass

# Seed für Reproduzierbarkeit
np.random.seed(42)
random.seed(42)

# Instanz für FakeXYZ
fake = Faker('de_DE')

# Generate Customer Profile

In [0]:
n_customers = 1000


first_names = ['Max', 'Anna', 'Lukas', 'Sophie', 'Tim', 'Laura', 'Felix', 'Emma', 
               'Paul', 'Mia', 'Leon', 'Hannah', 'Jonas', 'Lena', 'David', 
               'Sarah', 'Ben', 'Julia', 'Noah', 'Lisa']

last_names = ['Müller', 'Schmidt', 'Schneider', 'Fischer', 'Weber', 'Meyer', 
              'Wagner', 'Becker', 'Schulz', 'Hoffmann', 'Koch', 'Bauer',
              'Richter', 'Klein', 'Wolf', 'Schröder', 'Neumann', 'Schwarz',
              'Zimmermann', 'Braun']

cust_names = [f"{random.choice(first_names)} {random.choice(last_names)}" for i in range(n_customers)]

locations = []
for _ in range(n_customers):
    locations.append(fake.address())
addresses = pd.DataFrame(locations)

payment_methods = ['credit_card', 'bank_transfer', 'mailed_check', 'electronic_check']

customer_profile = pd.DataFrame({
    'customer_id': [f'CUST_{i:05d}' for i in range(1, n_customers + 1)],
    'customer_name': cust_names,
    'signup_date': pd.date_range(end='2026-03-01', periods=n_customers, freq='2D'),
    'plan_tier': np.random.choice(['Basic_50Mbps', 'Standard_200Mbps', 'Premium_1Gbps'], 
                                  n_customers, p=[0.35, 0.45, 0.2]),
    'address': addresses.values.flatten(),
    'contract_type': np.random.choice(['monthly', 'annual', '2-year'], 
                                     n_customers, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(payment_methods,
                                     n_customers, p=[0.2, 0.2, 0.25, 0.35]),
    'autopay_enabled': np.random.choice([True, False], n_customers, p=[0.7, 0.3])
})

customer_profile['account_age_months'] = (
    (pd.Timestamp('2026-03-01') - customer_profile['signup_date']).dt.days / 30
).round(1)

plan_prices = {'Basic_50Mbps': 49.99, 'Standard_200Mbps': 79.99, 'Premium_1Gbps': 119.99}
customer_profile['monthly_bill'] = customer_profile['plan_tier'].map(plan_prices)
discount_customers = np.random.choice([True, False], n_customers, p=[0.2, 0.8])
customer_profile.loc[discount_customers, 'monthly_bill'] *= 0.9


speed_tiers = {'Basic_50Mbps': 50, 'Standard_200Mbps': 200, 'Premium_1Gbps': 1000}
customer_profile['speed_tier_mbps'] = customer_profile['plan_tier'].map(speed_tiers)
customer_profile['data_usage_gb_last_month'] = np.random.exponential(300, n_customers).round(1)

print("Customer Profile created")

# Generate Chrun Labels

In [0]:
churn_rate = 0.15
n_churned = int(n_customers * churn_rate)
churned_customers = np.random.choice(customer_profile['customer_id'], n_churned, replace=False)

churn_labels = pd.DataFrame({
    'customer_id': customer_profile['customer_id'],
    'churned': customer_profile['customer_id'].isin(churned_customers).astype(int)
})

churn_reasons = ['competitor_price', 'poor_service', 'technical_issues', 'relocation', 'price_increase', 'unknown']
churn_labels.loc[churn_labels['churned'] == 1, 'churn_reason'] = np.random.choice(
    churn_reasons, n_churned, p=[0.35, 0.20, 0.20, 0.10, 0.05, 0.1]
)

print("Churn Labels erstellt")

# Generate Connection Logs 

In [0]:
connection_logs = []
log_id = 1

# Dictionary um technische Probleme pro Kunde zu tracken
customer_technical_issues = {}

for customer_id in customer_profile['customer_id']:
    customer_data = customer_profile[customer_profile['customer_id'] == customer_id].iloc[0]
    is_churned = churn_labels[churn_labels['customer_id'] == customer_id]['churned'].values[0]
    
    # Initialisiere Issue-Liste für diesen Kunden
    customer_technical_issues[customer_id] = []
    
    # Mehr Logs für Kunden mit Problemen
    n_logs = random.randint(20, 50)
    
    for i in range(n_logs):
        timestamp = datetime(2026, 3, 1) - timedelta(
            days=random.randint(1, 90),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )
        
        # Baseline Qualität (gut)
        speed_factor = random.uniform(0.85, 1.0)
        packet_loss = random.uniform(0, 2)
        latency = random.uniform(10, 40)
        downtime = 0
        drops = 0
        
        # Zufällige technische Probleme mit bestimmter Wahrscheinlichkeit
        problem_type = None
        
        # 20% Chance für technische Probleme (höher bei abgewanderten Kunden)
        problem_chance = 0.35 if is_churned else 0.15
        
        if random.random() < problem_chance:
            problem_type = np.random.choice([
                'slow_speed',
                'connection_drops',
                'high_latency',
                'outage',
                'packet_loss'
            ], p=[0.3, 0.15, 0.25, 0.1, 0.20])
            
            # Simuliere verschiedene Problemtypen
            if problem_type == 'slow_speed':
                speed_factor = random.uniform(0.2, 0.5)  # Nur 20-50% des Speeds
                latency = random.uniform(40, 100)
                
            elif problem_type == 'connection_drops':
                drops = random.randint(5, 25)
                packet_loss = random.uniform(5, 20)
                
            elif problem_type == 'high_latency':
                latency = random.uniform(100, 300)
                packet_loss = random.uniform(3, 10)
                
            elif problem_type == 'outage':
                downtime = random.randint(30, 180)
                speed_factor = 0
                drops = random.randint(10, 50)
                
            elif problem_type == 'packet_loss':
                packet_loss = random.uniform(10, 30)
                latency = random.uniform(60, 150)
            
            # Speichere dieses Problem-Event
            customer_technical_issues[customer_id].append({
                'timestamp': timestamp,
                'problem_type': problem_type,
                'log_id': f'LOG_{log_id:07d}',
                'severity': 'high' if downtime > 60 or speed_factor < 0.3 else 'medium'
            })
            

        connection_logs.append({
            'timestamp': timestamp,
            'issue_detected': 'error: ' + problem_type if problem_type else 'none',
            'customer_id': customer_id,
            'speed_measured_mbps': round(customer_data['speed_tier_mbps'] * speed_factor, 1),
            'packet_loss_percent': round(packet_loss, 2),
            'latency_ms': round(latency, 1),
            'downtime_minutes': downtime,
            'connection_drops_count': drops
        })
        
        log_id += 1

connection_quality_logs = pd.DataFrame(connection_logs)

print(f"Connection Logs erstellt: {len(connection_quality_logs)} Logs")
print(f"Davon mit Problemen: {len(connection_quality_logs[connection_quality_logs['issue_detected'] != 'none'])}")


# Generate Agents

In [0]:
n_agents = 20

agent_names = [f"{first_names[i]} {last_names[i]}" for i in range(n_agents)]

agent_profile = pd.DataFrame({
    'agent_id': [f'AGENT_{i:03d}' for i in range(1, n_agents + 1)],
    'agent_name': agent_names,
    'employment_date': pd.date_range(end='2026-03-01', periods=n_agents, freq='45D'), 
    'experience_level': np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], 
                                        n_agents, p=[0.3, 0.45, 0.2, 0.05])
})

# Berechne Beschäftigungsdauer in Monaten
agent_profile['employment_months'] = (
    (pd.Timestamp('2026-03-01') - agent_profile['employment_date']).dt.days / 30
).round(1)

# Gehalt basierend auf Experience Level und Department
salary_base = {
    'Junior': 2800,
    'Mid': 3500,
    'Senior': 4500,
    'Lead': 5500
}

agent_profile['monthly_salary_eur'] = agent_profile.apply(
    lambda row: salary_base[row['experience_level']] + 
                np.random.randint(-200, 300),
    axis=1
)

print("Agent Profile erstellt")

# Save Data

In [0]:
customer_profile.to_csv('customer_profile.csv', index=False)
churn_labels.to_csv('churn_label.csv', index=False)
connection_quality_logs.to_csv('connection_quality_log.csv', index=False)
agent_profile.to_csv('agent_profile.csv', index=False)

print(f"\nZusammenfassung:")
print(f"   • Customers: {len(customer_profile)}")
print(f"   • Churned Customers: {n_churned} ({churn_rate*100}%)")
print(f"   • Connection Quality Logs: {len(connection_quality_logs)}")
print(f"     - Mit erkannten Problemen: {(connection_quality_logs['issue_detected'] != 'none').sum()}")